<a href="https://colab.research.google.com/github/Antibote/pinn-eos-aluminum/blob/main/notebooks/pinn_eos_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PINN для уравнения состояния алюминия — общий блокнот команды

Кейс 02 (ML), хакатон «ИТ КоР 5.0». Репозиторий: https://github.com/Antibote/pinn-eos-aluminum · задачи: https://github.com/Antibote/pinn-eos-aluminum/issues

**Правила работы в блокноте**
1. `Среда выполнения → Сменить среду выполнения → GPU (T4)`.
2. Каждый раз после подключения запустите раздел **0. Подготовка** — он скачает данные и код.
3. Пишите только в **своём** разделе. Среда выполнения у каждого своя: переменные коллег у вас не появятся, пока вы сами не запустите их ячейки.
4. Общие результаты (модели, `normalization.json`, графики) сохраняйте в `ARTIFACTS` / `FIGURES` — они лежат на вашем Google Диске и не пропадут после отключения.
5. Готовый код переносим в `src/eos/` репозитория, чтобы финальная сдача запускалась через `python run.py`.

| Раздел | Задача | Ответственный |
|---|---|---|
| 1. Данные, EDA и бейзлайн | #1 | Деревянкин Кирилл |
| 2. Vanilla-модель | #2 | Елагин Матвей |
| 3. PINN и физические ограничения | #3 | Королева Ульяна |
| 4. Инференс, упаковка и сжатие | #4 | Володченко Александра |
| 5. Итоговые визуализации | #5 | Селивохин Ярослав |

## 0. Подготовка (общая, запускать каждому)

In [1]:
import torch
print('torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет — включите GPU в настройках среды')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch 2.11.0+cu130
GPU: Tesla T4


In [2]:
# Google Диск: сюда сохраняем модели и графики, чтобы они не пропали при отключении
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/pinn-eos')
except ImportError:  # запуск не в Colab
    WORK = Path('pinn-eos-work')
ARTIFACTS = WORK / 'artifacts'
FIGURES = WORK / 'figures'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
print('Артефакты:', ARTIFACTS)

Mounted at /content/drive
Артефакты: /content/drive/MyDrive/pinn-eos/artifacts


In [3]:
# Код репозитория (src/eos) и данные с Яндекс Диска
import json, subprocess, sys, urllib.parse, urllib.request, zipfile

REPO = Path('pinn-eos-aluminum')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Antibote/pinn-eos-aluminum.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))

DATA = REPO / 'data'
if not list(DATA.glob('Obl*.txt')):
    api = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?' + urllib.parse.urlencode(
        {'public_key': 'https://disk.yandex.ru/d/iUSbYrJcsbXSbw', 'path': '/Материалы к хакатону.zip'})
    href = json.load(urllib.request.urlopen(api))['href']
    urllib.request.urlretrieve(href, 'data.zip')
    zipfile.ZipFile('data.zip').extractall(DATA)
print(len(list(DATA.glob('Obl*.txt'))), 'файлов данных в', DATA)

20 файлов данных в pinn-eos-aluminum/data


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from eos.data import load_all, region_grid

df = load_all(DATA)
print(df.shape)
df.head()

(1130053, 5)


,rho,T,P,E,region
0,0.001000,0.1,0.000515,-46.797895,Obl01
1,0.001030,0.1,0.000540,-46.790300,Obl01
2,0.001060,0.1,0.000569,-46.782552,Obl01
3,0.001091,0.1,0.000597,-46.774662,Obl01
4,0.001124,0.1,0.000626,-46.766630,Obl01


---
## 1. Данные, EDA и бейзлайн — Деревянкин Кирилл (#1)

- [ ] EDA: распределения, проверка стыков областей
- [ ] Сечения P и E при 3–4 фиксированных ρ и T
- [ ] Нормировка → `ARTIFACTS / 'normalization.json'`
- [ ] Разбиение train/val/test (сохранить индексы или seed)
- [ ] Бейзлайн: билинейная интерполяция, точность и время на 10⁵ запросов

### 1.1 EDA

In [ ]:
# ваш код

123123123

### 1.2 Сечения

In [ ]:
# ваш код

### 1.3 Нормировка и разбиение

In [ ]:
# ваш код

### 1.4 Бейзлайн

In [ ]:
# ваш код

---
## 2. Vanilla-модель — Елагин Матвей (#2)

- [ ] MLP на PyTorch (ρ, T → P, E) на `DEVICE`
- [ ] Кривые обучения
- [ ] Производные через autograd
- [ ] Тепловая карта ошибок и карта нарушений
- [ ] Сохранить веса в `ARTIFACTS / 'vanilla.pt'`

### 2.1 Модель и обучение

In [ ]:
# ваш код

### 2.2 Производные через autograd

In [ ]:
# ваш код

### 2.3 Ошибки и нарушения

In [ ]:
# ваш код

---
## 3. PINN и физические ограничения — Королева Ульяна (#3)

- [ ] Физические штрафы: согласованность, ∂P/∂ρ > 0, ∂E/∂T ≥ 0 (в данных при T ≲ 7 K плато), опц. 0 < c_s < c
- [ ] Обучение PINN, подбор весов λ
- [ ] Сравнение с Vanilla
- [ ] Сохранить веса в `ARTIFACTS / 'pinn.pt'`

### 3.1 Функция потерь

In [ ]:
# ваш код

### 3.2 Обучение PINN

In [ ]:
# ваш код

### 3.3 Сравнение с Vanilla

In [ ]:
# ваш код

---
## 4. Инференс, упаковка и сжатие — Володченко Александра (#4)

- [ ] Класс `EOSModel`: `predict`, `sound_speed`, `check_physics`
- [ ] Размер модели (≤ 10 МБ), время на 10⁵ запросов (CPU и GPU)
- [ ] Экспорт TorchScript / ONNX
- [ ] Сжатие: квантизация / меньшая сеть
- [ ] Единая таблица сравнения baseline / Vanilla / PINN

### 4.1 Класс-обёртка

In [ ]:
# ваш код

### 4.2 Размер и скорость

In [ ]:
# ваш код

### 4.3 Экспорт

In [ ]:
# ваш код

### 4.4 Сжатие

In [ ]:
# ваш код

### 4.5 Таблица сравнения

In [ ]:
# ваш код

---
## 5. Итоговые визуализации — Селивохин Ярослав (#5)

- [ ] Графики для документации и презентации → `FIGURES`

### 5.1 Графики для отчёта

In [ ]:
# ваш код